In [1]:
import pandas as pd
import numpy as np
from pvlib import solarposition
from skyfield.api import load, Topos, EarthSatellite
from astral import LocationInfo
from astral.sun import sun
import pvlib
from datetime import datetime
import os

import helper_code.dataloading as dataloading
import helper_code.model_functions as model_functions
import helper_code.data_vis as data_vis

# reload imported code
import importlib
importlib.reload(data_vis)
importlib.reload(dataloading)

<module 'helper_code.dataloading' from '/Users/sharonni/Desktop/alertcalifornia-anomaly-detection/helper_code/dataloading.py'>

In [86]:
triplet_embeddings = pd.read_csv("viewer_data/reduced_embeddings_triplet.csv")
triplet_labels = pd.read_csv("viewer_data/labels_triplet.csv")
triplet = triplet_embeddings.join(triplet_labels)

In [87]:
hsad_embeddings = pd.read_csv("viewer_data/reduced_embeddings_hsad.csv")
hsad_labels = pd.read_csv("viewer_data/labels_hsad.csv")
hsad = hsad_embeddings.join(hsad_labels)
hsad["id"] = hsad["annotation_ids"]

In [88]:
deepsad_embeddings = pd.read_csv("viewer_data/reduced_embeddings_deepsad.csv")
deepsad_labels = pd.read_csv("viewer_data/labels_deepsad.csv")
deepsad = deepsad_embeddings.join(deepsad_labels)
deepsad["id"] = deepsad["annotation_ids"]

In [89]:
final_embeddings = pd.read_csv("viewer_data/reduced_embeddings_final.csv")
final_labels = pd.read_csv("viewer_data/labels_final.csv")
final = final_embeddings.join(final_labels)

In [90]:
def add_features(df, lat, lon, save_csv=False, dataset_name=None):

    df = df.copy()

    df['timestamp'] = df['img_urls'].str.extract(r"https:\/\/tools\.alertcalifornia\.org\/fireframes5\/digitalpath-redis\/[^\/]+\/\d{4}\/\d{3}\/\d{2}\/(\d+)\.")

    df['datetime_local'] = (pd.to_datetime(df['timestamp'], unit='s', utc=True)
        .dt.tz_convert('America/Los_Angeles')
                           )

    df['datetime_UTC'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

    # solar altitude
    solar_alt = pvlib.solarposition.get_solarposition(time=df['datetime_local'],
                                                       latitude=lat,
                                                       longitude=lon
                                                      )['apparent_elevation']
    df['solar_altitude'] = np.array(solar_alt)

    # lunar altitude
    eph = load('de421.bsp')
    ts = load.timescale()
    
    earth = eph['earth']
    moon = eph['moon']
    
    location = earth + Topos(
        latitude_degrees=lat,
        longitude_degrees=lon
    )

    utc_times = df['datetime_UTC'].dt.to_pydatetime()
    sf_times = ts.from_datetimes(utc_times)

    apparent = location.at(sf_times).observe(moon).apparent()
    alt, az, distance = apparent.altaz()
    
    df['lunar_altitude'] = alt.degrees

    # sunrise & sunset
    loc = LocationInfo(latitude=lat, longitude=lon)
    df['date'] = df['datetime_local'].dt.date
    daily_sun = df.groupby('date')['datetime_local'].first().apply(
        lambda dt: sun(loc.observer, date=dt, tzinfo=dt.tz)
    )
    df['sunrise'] = df['date'].map(lambda d: daily_sun[d]['sunrise'])
    df['sunset'] = df['date'].map(lambda d: daily_sun[d]['sunset'])

    df['daytime'] = (df['datetime_local'] > df['sunrise']) & \
    (df['datetime_local'] < df['sunset'])

    # hour & month
    df['hour'] = df['datetime_local'].dt.hour
    df['month'] = df['datetime_local'].dt.month

    # season
    def get_season(month):
        if month in [12, 1, 2]:
            return 'Winter'
        elif month in [3, 4, 5]:
            return 'Spring'
        elif month in [6, 7, 8]:
            return 'Summer'
        else:
            return 'Fall'
    df['season'] = df['month'].apply(get_season)

    pc_names = {"img_urls": "img_url", "ids": "id", "labels": "label"}
    for col in df.columns:
        if col in [str(i) for i in range(10)]:
            pc_names[col] = f"PC{int(col)+1}"

    df.rename(columns=pc_names, inplace=True)

    if save_csv:
        name = f"{dataset_name}_pca_features.csv"
        df.to_csv(name, index=False)
        print(f"saved csv file: {name}")

    return df

In [91]:
triplet_features = add_features(triplet, lat=33.1, lon=-117.146, save_csv=True, dataset_name="triplet")

/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_2505/1286513177.py:7: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['datetime_local'] = (pd.to_datetime(df['timestamp'], unit='s', utc=True)
/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_2505/1286513177.py:11: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['datetime_UTC'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_2505/1286513177.py:32: 

saved csv file: triplet_pca_features.csv


In [92]:
hsad_features = add_features(hsad, lat=33.1, lon=-117.146, save_csv=True, dataset_name="hsad")

/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_2505/1286513177.py:7: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['datetime_local'] = (pd.to_datetime(df['timestamp'], unit='s', utc=True)
/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_2505/1286513177.py:11: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['datetime_UTC'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_2505/1286513177.py:32: 

saved csv file: hsad_pca_features.csv


In [93]:
deepsad_features = add_features(deepsad, lat=33.1, lon=-117.146, save_csv=False, dataset_name="deepsad")

/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_2505/1286513177.py:7: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['datetime_local'] = (pd.to_datetime(df['timestamp'], unit='s', utc=True)
/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_2505/1286513177.py:11: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['datetime_UTC'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_2505/1286513177.py:32: 

In [94]:
final_features = add_features(final, lat=33.1, lon=-117.146, save_csv=True, dataset_name="final")

/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_2505/1286513177.py:7: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['datetime_local'] = (pd.to_datetime(df['timestamp'], unit='s', utc=True)
/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_2505/1286513177.py:11: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['datetime_UTC'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_2505/1286513177.py:32: 

saved csv file: final_pca_features.csv


In [95]:
label_map = {
    0: {"name": "Normal", "color": "#00d6c7"}, 
    1: {"name": "Abormal", "color": "#c69214"},  
}

In [96]:
data_vis.save_embeddings(
    triplet_features,
    "Coronado Hills (Triplet)",
    metadata={},
    label_map=label_map,
    output_path="embedding_data/embeddings1.json")

saved dataset 'Coronado Hills (Triplet)' to embedding_data/embeddings1.json


In [97]:
data_vis.save_embeddings(
    deepsad_features,
    "Coronado Hills (DeepSAD)",
    metadata={},
    label_map=label_map,
    output_path="embedding_data/embeddings1.json")

saved dataset 'Coronado Hills (DeepSAD)' to embedding_data/embeddings1.json


In [98]:
label_map = {
    0: {"name": "Level 1-Clear", "color": "#00d6c7"}, 
    1: {"name": "Level 2-Minor", "color": "#6e963b"},  
    2: {"name": "Level 3-Major", "color": "#1e1d1f"},  
    3: {"name": "Level 4-Severe", "color": "#c69214"}
}

data_vis.save_embeddings(
    hsad_features,
    "Coronado Hills (H-SAD)",
    metadata={},
    label_map=label_map,
    output_path="embedding_data/embeddings1.json")

saved dataset 'Coronado Hills (H-SAD)' to embedding_data/embeddings1.json


In [99]:
label_map = {
    0: {"name": "Normal", "color": "#00d6c7"}, 
    1: {"name": "Abormal", "color": "#c69214"},  
}

data_vis.save_embeddings(
    final_features,
    "Final Multi-Camera Model (Triplet)",
    metadata={},
    label_map=label_map,
    output_path="embedding_data/embeddings1.json")

saved dataset 'Final Multi-Camera Model (Triplet)' to embedding_data/embeddings1.json


In [203]:
# total variance of original data
total_var = final_embeddings.values.var(axis=0).sum()

# variance per PC
pc_var = final_embeddings.var(axis=0)

explained_ratio = pc_var / total_var

test = pd.DataFrame({
    "PC": final_embeddings.columns,
    "ExplainedVarianceRatio": explained_ratio
})

test

,PC,ExplainedVarianceRatio
0,0,0.872966
1,1,0.030582
2,2,0.022350
3,3,0.017364
4,4,0.013373
5,5,0.012458
6,6,0.009053
7,7,0.008186
8,8,0.007659
9,9,0.006275


In [66]:
palomar = df3.join(df4)
palomar['timestamp'] = palomar['img_urls'].str.extract(r"https:\/\/tools\.alertcalifornia\.org\/fireframes5\/digitalpath-redis\/[^\/]+\/\d{4}\/\d{3}\/\d{2}\/(\d+)\.")

In [133]:
hsad_features

,PC1,PC2,PC3,PC4,label,img_url,annotation_ids,id,timestamp,datetime_local,datetime_UTC,solar_altitude,lunar_altitude,date,sunrise,sunset,daytime,hour,month,season
0,-9.178001,-1.117732,-0.038978,0.008658,3,https://tools.alertcalifornia.org/fireframes5/...,1084477,1084477,1739509466,2025-02-13 21:04:26-08:00,2025-02-14 05:04:26+00:00,-44.742613,25.892380,2025-02-13,2025-02-13 06:33:52.549474-08:00,2025-02-13 17:32:04.876333-08:00,False,21,2,winter
1,2.493089,0.013392,0.567745,-0.191358,1,https://tools.alertcalifornia.org/fireframes5/...,1088551,1088551,1758013260,2025-09-16 02:01:00-07:00,2025-09-16 09:01:00+00:00,-49.979219,7.760813,2025-09-16,2025-09-16 06:33:17.877204-07:00,2025-09-16 18:52:41.777126-07:00,False,2,9,fall
2,5.040985,-0.317855,0.339538,0.416977,0,https://tools.alertcalifornia.org/fireframes5/...,1082316,1082316,1731387955,2024-11-11 21:05:55-08:00,2024-11-12 05:05:55+00:00,-53.759065,53.703843,2024-11-11,2024-11-11 06:16:29.994784-08:00,2024-11-11 16:48:28.445171-08:00,False,21,11,fall
3,3.564076,0.094514,-0.260990,-0.451172,1,https://tools.alertcalifornia.org/fireframes5/...,1088448,1088448,1757642700,2025-09-11 19:05:00-07:00,2025-09-12 02:05:00+00:00,-1.940802,-23.608273,2025-09-11,2025-09-11 06:30:02.264125-07:00,2025-09-11 18:59:30.456447-07:00,False,19,9,fall
4,-7.830932,0.619472,-0.444600,-0.170822,3,https://tools.alertcalifornia.org/fireframes5/...,1088772,1088772,1758808891,2025-09-25 07:01:31-07:00,2025-09-25 14:01:31+00:00,4.061182,-38.580601,2025-09-25,2025-09-25 06:39:14.539615-07:00,2025-09-25 18:40:24.294582-07:00,True,7,9,fall
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1383,3.387612,0.235587,-0.140524,-0.266369,1,https://tools.alertcalifornia.org/fireframes5/...,1081795,1081795,1729278152,2024-10-18 12:02:32-07:00,2024-10-18 19:02:32+00:00,46.282328,-33.366991,2024-10-18,2024-10-18 06:55:57.496726-07:00,2024-10-18 18:10:41.997725-07:00,True,12,10,fall
1384,-9.059099,-1.093052,-0.023289,-0.077556,3,https://tools.alertcalifornia.org/fireframes5/...,1086472,1086472,1747998132,2025-05-23 04:02:12-07:00,2025-05-23 11:02:12+00:00,-18.615580,10.502462,2025-05-23,2025-05-23 05:44:12.393402-07:00,2025-05-23 19:46:56.520967-07:00,False,4,5,spring
1385,5.811303,-0.407402,-0.555504,0.397848,0,https://tools.alertcalifornia.org/fireframes5/...,1085748,1085748,1744232592,2025-04-09 14:03:12-07:00,2025-04-09 21:03:12+00:00,59.657330,-29.347708,2025-04-09,2025-04-09 06:25:44.795893-07:00,2025-04-09 19:14:45.578029-07:00,True,14,4,spring
1386,-10.280075,-0.584701,-0.051992,0.454940,3,https://tools.alertcalifornia.org/fireframes5/...,1083375,1083375,1735376430,2024-12-28 01:00:30-08:00,2024-12-28 09:00:30+00:00,-71.696771,-45.992118,2024-12-28,2024-12-28 06:51:09.653009-08:00,2024-12-28 16:49:56.786069-08:00,False,1,12,winter


In [68]:
df = add_features(palomar, lat=33.36, lon=-116.87, save_csv=True, cam_name="palomar_obs_1")

saved csv file to camera_data/palomar_obs_1/pca_features.csv


/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_30604/2206395373.py:5: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['datetime_local'] = (pd.to_datetime(df['timestamp'], unit='s', utc=True)
/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_30604/2206395373.py:9: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['datetime_UTC'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_30604/2206395373.py:30

In [69]:
data_vis.save_embeddings(df,
    "palomar_obs_1",
    label_map=label_map,
    output_path="embedding_data/embeddings_test.json")

saved dataset 'palomar_obs_1' to embedding_data/embeddings_test.json


In [70]:
df5 = pd.read_csv("labels_urls_ids_mesa_grande.csv")
df6 = pd.read_csv("reduced_embeddings_mesa_grande.csv")

In [71]:
mesa_grande_n = df5.join(df6)
mesa_grande_n['timestamp'] = mesa_grande_n['img_urls'].str.extract(r"https:\/\/tools\.alertcalifornia\.org\/fireframes5\/digitalpath-redis\/[^\/]+\/\d{4}\/\d{3}\/\d{2}\/(\d+)\.")

In [72]:
mesa_grande_n

,labels,img_urls,annotation_ids,0,1,2,3,4,timestamp
0,0,https://tools.alertcalifornia.org/fireframes5/...,55727,5.471479,0.007922,1.240778,0.845213,1.465930,1732190539
1,0,https://tools.alertcalifornia.org/fireframes5/...,56150,3.449308,0.979847,0.057175,-0.059955,1.554728,1757152878
2,0,https://tools.alertcalifornia.org/fireframes5/...,53278,3.523902,2.859374,0.520055,0.806442,0.457418,1742461302
3,0,https://tools.alertcalifornia.org/fireframes5/...,52259,4.482541,-2.066068,-0.309867,0.064704,-0.080141,1756735331
4,0,https://tools.alertcalifornia.org/fireframes5/...,55782,4.913761,2.968381,1.926073,-0.305259,0.198810,1728302519
...,...,...,...,...,...,...,...,...,...
378,0,https://tools.alertcalifornia.org/fireframes5/...,56256,3.730495,0.655744,1.004255,2.596185,-1.934280,1749783926
379,0,https://tools.alertcalifornia.org/fireframes5/...,54714,6.125154,-3.305253,0.098783,0.631595,-0.369025,1728835324
380,0,https://tools.alertcalifornia.org/fireframes5/...,54635,4.982977,2.106985,2.295540,0.175496,0.686929,1740571284
381,0,https://tools.alertcalifornia.org/fireframes5/...,53518,-0.020257,6.234782,-1.256226,-1.561303,0.421649,1753254034


In [73]:
df = add_features(mesa_grande_n, lat=33.19, lon=-116.76, save_csv=True, cam_name="mesa_grande_n")

saved csv file to camera_data/mesa_grande_n/pca_features.csv


/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_30604/2206395373.py:5: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['datetime_local'] = (pd.to_datetime(df['timestamp'], unit='s', utc=True)
/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_30604/2206395373.py:9: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['datetime_UTC'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
/var/folders/d9/1v24rcz14jqcmc6p7g6hmkdm0000gn/T/ipykernel_30604/2206395373.py:30

In [204]:
data_vis.save_embeddings(df,
    "mesa_grande_n",
    label_map=label_map,
    output_path="embedding_data/embeddings1.json")

NameError: name 'df' is not defined

In [4]:
import json
import csv

def save_unique_img_urls_with_ids(json_path, output_csv):
    with open(json_path) as f:
        data = json.load(f)

    url_to_id = {}

    for dataset in data.get("datasets", []):
        for point in dataset.get("points", []):
            url = point.get("img_url")
            pid = point.get("id")

            if url and url not in url_to_id:
                url_to_id[url] = pid

    with open(output_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["img_url", "id"])
        for url, pid in url_to_id.items():
            writer.writerow([url, pid])

    print(f"Saved {len(url_to_id)} unique URLs to {output_csv}")


# usage
save_unique_img_urls_with_ids(
    "embedding_data/embeddings1.json",
    "embedding_img_urls.csv"
)

Saved 4267 unique URLs to embedding_img_urls.csv
